# Job Raider - Example Usage

This notebook demonstrates how to use the Job Raider pipeline for automated job application processing.

**Author:** Job Raider  
**Date:** 2026-04-21

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
# Add src to path
import sys
from pathlib import Path

# Ensure we're running from the project root
if Path.cwd().name != 'job-raider':
    # Try to navigate to project root
    import os
    os.chdir(Path.cwd().parent.parent)

sys.path.insert(0, str(Path.cwd() / 'src'))

# Import Job Raider modules
from pipeline.orchestrator import PipelineOrchestrator, PipelineConfig
from models.user_profile import (
    UserProfile,
    ContactInfo,
    Skill,
    SkillCategory,
    ProficiencyLevel,
    Project,
    TargetJob,
    ExperienceLevel,
)
from scrapers.storage import JobListingStorage
from scoring.matcher import JobMatcher
from generation.selector import ResumeSelector
from metrics.cost_tracker import CostTracker
from metrics.outcome_tracker import OutcomeTracker

print("Job Raider modules imported successfully!")

## Creating a User Profile

Let's create a sample user profile. In production, you would load this from a resume file.

In [ ]:
# Create a user profile
profile = UserProfile(
    contact_info=ContactInfo(
        name="Jane Developer",
        email="jane.developer@example.com",
        phone="555-1234",
        linkedin="https://linkedin.com/in/janedeveloper",
        github="https://github.com/janedeveloper",
    ),
    skills={
        SkillCategory.PROGRAMMING: [
            Skill(name="Python", proficiency=ProficiencyLevel.EXPERT, years_experience=7),
            Skill(name="JavaScript", proficiency=ProficiencyLevel.INTERMEDIATE, years_experience=4),
            Skill(name="TypeScript", proficiency=ProficiencyLevel.INTERMEDIATE, years_experience=3),
        ],
        SkillCategory.FRAMEWORKS: [
            Skill(name="Django", proficiency=ProficiencyLevel.EXPERT, years_experience=6),
            Skill(name="FastAPI", proficiency=ProficiencyLevel.ADVANCED, years_experience=3),
            Skill(name="React", proficiency=ProficiencyLevel.INTERMEDIATE, years_experience=4),
        ],
        SkillCategory.DATABASE: [
            Skill(name="PostgreSQL", proficiency=ProficiencyLevel.ADVANCED, years_experience=6),
            Skill(name="MongoDB", proficiency=ProficiencyLevel.INTERMEDIATE, years_experience=3),
        ],
        SkillCategory.CLOUD: [
            Skill(name="AWS", proficiency=ProficiencyLevel.ADVANCED, years_experience=4),
            Skill(name="Docker", proficiency=ProficiencyLevel.INTERMEDIATE, years_experience=3),
        ],
    },
    experience=[
        {
            "title": "Senior Python Engineer",
            "company": "Tech Innovations Inc",
            "start_date": "2021-01-01",
            "end_date": "2024-12-31",
            "description": "Led development of microservices architecture serving 1M+ users",
        },
        {
            "title": "Python Developer",
            "company": "StartUp Co",
            "start_date": "2018-06-01",
            "end_date": "2020-12-31",
            "description": "Built RESTful APIs and data processing pipelines",
        },
    ],
    projects=[
        Project(
            name="E-commerce Platform",
            description="Full-stack e-commerce platform with payment processing",
            technologies=["Python", "Django", "React", "PostgreSQL", "Redis", "Docker"],
            start_date="2022-01-01",
            end_date="2023-06-30",
            highlights=[
                "Handled 50,000+ daily active users",
                "Implemented caching reducing load times by 60%",
                "Integrated Stripe payment processing",
            ],
        ),
        Project(
            name="Real-time Analytics Dashboard",
            description="Dashboard for monitoring system metrics in real-time",
            technologies=["Python", "FastAPI", "WebSockets", "InfluxDB", "Grafana"],
            start_date="2021-06-01",
            end_date="2022-12-31",
            highlights=[
                "Processed 1M+ events per day",
                "Reduced alert response time by 80%",
            ],
        ),
    ],
    education=[
        {
            "degree": "B.S. Computer Science",
            "school": "University of California",
            "graduation_year": "2018",
        }
    ],
    target_job=TargetJob(
        keywords=["python", "engineer", "developer", "backend", "software"],
        locations=["remote", "san francisco", "new york", "austin"],
        experience_levels=[ExperienceLevel.MID, ExperienceLevel.SENIOR],
    ),
)

print(f"Profile created for: {profile.contact_info.name}")
print(f"Years of experience: {profile.years_of_experience}")
print(f"Total skills: {sum(len(skills) for skills in profile.skills.values())}")
print(f"Projects: {len(profile.projects)}")

## Scraping Job Listings

Let's scrape some job listings from various platforms.

In [ ]:
# Configure scraping parameters
config = PipelineConfig(
    keywords=["python", "engineer"],
    locations=["remote"],
    sources=["linkedin", "indeed"],  # Only use these sources
    dry_run=True,  # No actual submissions
    skip_submission=True,  # Skip submission stage
    max_jobs_to_present=10,  # Only show top 10
)

# Create orchestrator
orchestrator = PipelineOrchestrator(
    config=config,
    user_profile=profile,
)

# Run pipeline (skip scraping in notebook, use existing data)
print("Pipeline configured successfully!")
print(f"Keywords: {config.keywords}")
print(f"Locations: {config.locations}")
print(f"Sources: {config.sources}")

## Scoring Jobs

Let's score some sample job listings to see how well they match our profile.

In [ ]:
# Create sample job listings
from models.job_listing import JobListing, JobSource, SalaryRange
from datetime import datetime

# Sample jobs
jobs = [
    JobListing(
        title="Senior Python Backend Engineer",
        company="TechCorp",
        location="Remote",
        description="We are looking for a senior Python engineer...",
        requirements=[
            "7+ years of Python experience",
            "Experience with Django and FastAPI",
            "Knowledge of AWS services",
        ],
        responsibilities=[
            "Design and implement scalable APIs",
            "Mentor junior developers",
        ],
        skills=["python", "django", "fastapi", "aws", "postgresql"],
        salary_range=SalaryRange(
            min_amount=150000,
            max_amount=200000,
            currency="USD",
            period="annual",
        ),
        source=JobSource.LINKEDIN,
        job_id="linkedin_12345",
        posted_date=datetime.now(),
    ),
    JobListing(
        title="Full Stack Developer",
        company="DataInc",
        location="San Francisco, CA",
        description="Full stack developer role...",
        requirements=[
            "5+ years of experience",
            "React and Python",
        ],
        skills=["python", "javascript", "react", "django"],
        source=JobSource.INDEED,
        job_id="indeed_67890",
        posted_date=datetime.now(),
    ),
]

# Score jobs
matcher = JobMatcher()

print("Job Scoring Results:")
print("=" * 60)

for job in jobs:
    score = matcher.match_and_score(job, profile)
    
    print(f"\n{job.title} at {job.company}")
    print(f"  Total Score: {score.total_score}/100")
    print(f"  - Keywords: {score.keyword_score}/30")
    print(f"  - Skills: {score.skills_score}/40")
    print(f"  - Experience: {score.experience_score}/20")
    print(f"  - Location: {score.location_score}/10")
    print(f"  Location: {job.location}")
    if job.salary_range:
        print(f"  Salary: ${job.salary_range.min_amount:,.0f} - ${job.salary_range.max_amount:,.0f}")

## Resume Content Selection

Let's select the most relevant projects and keywords for a specific job.

In [ ]:
# Select content for the first job
job = jobs[0]  # Senior Python Backend Engineer

print(f"Selecting content for: {job.title} at {job.company}")
print("=" * 60)

# NOTE: This would normally call Ollama
# For demonstration, we'll show what the selection would include

print("\nSelected Projects (simulated):")
for i, project in enumerate(profile.projects[:3], 1):
    print(f"  {i}. {project.name}")
    print(f"     Technologies: {', '.join(project.technologies[:3])}")

print("\nTarget Keywords (simulated):")
target_keywords = job.skills[:5]
for kw in target_keywords:
    print(f"  - {kw}")

## Cost Tracking

Let's see how much it would cost to process multiple applications.

In [ ]:
# Create cost tracker
cost_tracker = CostTracker()

# Get cost estimates
print("Cost Estimates:")
print("=" * 60)

# Estimate for 50 applications with local models
estimate_local = cost_tracker.get_cost_estimate(
    num_jobs=50,
    use_local_models=True,
)

print(f"\n50 applications with local models (Ollama):")
print(f"  Total cost: ${estimate_local['total_cost_usd']:.2f}")
print(f"  Cost per application: ${estimate_local['cost_per_application']:.4f}")

# Estimate for 50 applications with API
estimate_api = cost_tracker.get_cost_estimate(
    num_jobs=50,
    use_local_models=False,
)

print(f"\n50 applications with API (Anthropic):")
print(f"  Total cost: ${estimate_api['total_cost_usd']:.2f}")
print(f"  Cost per application: ${estimate_api['cost_per_application']:.4f}")

savings = estimate_api['total_cost_usd'] - estimate_local['total_cost_usd']
print(f"\nSavings with local models: ${savings:.2f} ({savings/estimate_api['total_cost_usd']*100:.1f}%)")

## Outcome Tracking

Let's see how to track application outcomes.

In [ ]:
# Create outcome tracker
outcome_tracker = OutcomeTracker()

# Track a sample application
app_id = outcome_tracker.track_application(
    application_id="app_001",
    job_title="Senior Python Backend Engineer",
    company="TechCorp",
)

print(f"Tracked application: {app_id}")

# Update status
outcome_tracker.update_status(
    application_id="app_001",
    status="under_review",
    note="Application submitted via Easy Apply",
)

# Add interview
outcome_tracker.add_interview(
    application_id="app_001",
    stage="screening",
    scheduled_date=datetime.now(),
)

# Get application details
app = outcome_tracker.get_application("app_001")

print(f"\nApplication Status: {app.current_status}")
print(f"Interviews scheduled: {len(app.interviews)}")

# Get conversion metrics
metrics = outcome_tracker.get_conversion_metrics(days=30)

print(f"\nConversion Metrics (30 days):")
print(f"  Total applications: {metrics.total_applications}")
print(f"  Screening rate: {metrics.screening_rate:.1%}")
print(f"  Offer rate: {metrics.offer_rate:.1%}")

## Summary

This notebook demonstrated the key components of Job Raider:

1. **User Profile**: Created a detailed profile with skills, experience, and projects
2. **Job Scraping**: Configured pipeline parameters for job search
3. **Job Scoring**: Scored jobs against the profile using the 100-point heuristic
4. **Resume Selection**: Selected relevant projects and keywords for tailoring
5. **Cost Tracking**: Estimated costs for processing applications
6. **Outcome Tracking**: Tracked application status and conversion metrics

### Next Steps

- Run the full pipeline with `python main.py --interactive`
- Generate tailored resumes with `--no-dry-run`
- Track outcomes and analyze conversion rates
- Optimize scoring thresholds based on results